In [1]:
import numpy as np

from src.preprocessing.loader import DataLoader
from src.preprocessing.cleaner import DataCleaner
from src.preprocessing.encoder import DataEncoder
from src.preprocessing.scaler import DataScaler
from src.preprocessing.splitter import DataSplitter

from src.models.isolation_forest import IsolationForestModel
from src.config.isolation_forest_config import IsolationForestConfig

In [2]:
from src.preprocessing.loader import DataLoader

loader = DataLoader()

files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv", 
]

df = loader.load_multiple(files)

print(df.shape)

2026-07-28 15:05:54 | INFO     | AdaptiveRL | Loading Monday-WorkingHours.pcap_ISCX.csv started.
2026-07-28 15:05:54 | INFO     | AdaptiveRL | Reading /home/kalpe/projects/adaptive_rl_anomaly_detection/datasets/raw/Monday-WorkingHours.pcap_ISCX.csv
2026-07-28 15:05:56 | INFO     | AdaptiveRL | Loaded Monday-WorkingHours.pcap_ISCX.csv | Shape=(529918, 79)
2026-07-28 15:05:56 | INFO     | AdaptiveRL | Loading Monday-WorkingHours.pcap_ISCX.csv completed in 2.5829 seconds.
2026-07-28 15:05:56 | INFO     | AdaptiveRL | Loading Tuesday-WorkingHours.pcap_ISCX.csv started.
2026-07-28 15:05:56 | INFO     | AdaptiveRL | Reading /home/kalpe/projects/adaptive_rl_anomaly_detection/datasets/raw/Tuesday-WorkingHours.pcap_ISCX.csv
2026-07-28 15:05:58 | INFO     | AdaptiveRL | Loaded Tuesday-WorkingHours.pcap_ISCX.csv | Shape=(445909, 79)
2026-07-28 15:05:58 | INFO     | AdaptiveRL | Loading Tuesday-WorkingHours.pcap_ISCX.csv completed in 1.9650 seconds.
2026-07-28 15:05:58 | INFO     | AdaptiveRL | Lo

(2830743, 79)


In [3]:
from src.preprocessing.cleaner import DataCleaner

cleaner = DataCleaner()

df = cleaner.clean(df)

print(df.shape)

2026-07-28 15:06:10 | INFO     | AdaptiveRL | Replacing Infinite Values started.
2026-07-28 15:06:12 | INFO     | AdaptiveRL | Replacing Infinite Values completed in 2.4423 seconds.
2026-07-28 15:06:12 | INFO     | AdaptiveRL | Removing Duplicates started.
2026-07-28 15:06:23 | INFO     | AdaptiveRL | Removed 308381 duplicate rows.
2026-07-28 15:06:23 | INFO     | AdaptiveRL | Removing Duplicates completed in 10.9877 seconds.
2026-07-28 15:06:23 | INFO     | AdaptiveRL | Removing Missing Values started.
2026-07-28 15:06:24 | INFO     | AdaptiveRL | Removed 1564 rows containing missing values.
2026-07-28 15:06:24 | INFO     | AdaptiveRL | Removing Missing Values completed in 0.9148 seconds.
2026-07-28 15:06:24 | INFO     | AdaptiveRL | Removing Constant Columns started.
2026-07-28 15:06:26 | INFO     | AdaptiveRL | Removed 8 constant columns.
2026-07-28 15:06:26 | INFO     | AdaptiveRL | Removing Constant Columns completed in 2.0341 seconds.
2026-07-28 15:06:26 | INFO     | AdaptiveRL |

(2520798, 71)


In [4]:
df.columns = df.columns.str.strip()

In [5]:
from src.preprocessing.encoder import DataEncoder

encoder = DataEncoder(target_column=" Label")

df = encoder.fit_transform(df)

print(df.dtypes["Label"])
print(df["Label"].unique()[:10])

2026-07-28 15:06:30 | INFO     | AdaptiveRL | Encoding Dataset started.
2026-07-28 15:06:31 | INFO     | AdaptiveRL | Encoding Dataset completed in 0.2864 seconds.
2026-07-28 15:06:31 | INFO     | AdaptiveRL | Encoding completed.


int64
[ 0  7 11  6  5  4  3  8 12 14]


In [6]:
from src.preprocessing.scaler import DataScaler

scaler = DataScaler(
    method="standard",
    target_column="Label",
)

df = scaler.fit_transform(df)

print(df.shape)
print(df["Label"].dtype)
print(df["Label"].unique()[:10])

2026-07-28 15:06:36 | INFO     | AdaptiveRL | Scaler (standard) fitted on 70 feature columns.
2026-07-28 15:06:37 | INFO     | AdaptiveRL | Scaling Dataset started.
2026-07-28 15:06:39 | INFO     | AdaptiveRL | Scaling Dataset completed in 2.4900 seconds.
2026-07-28 15:06:39 | INFO     | AdaptiveRL | Scaling completed.


(2520798, 71)
int64
[ 0  7 11  6  5  4  3  8 12 14]


In [7]:
X = df.drop(columns=["Label"])
y = df["Label"]

print(f"Features shape : {X.shape}")
print(f"Labels shape   : {y.shape}")
print("\nLabel Distribution:")
print(y.value_counts().sort_index())

Features shape : (2520798, 70)
Labels shape   : (2520798,)

Label Distribution:
Label
0     2095057
1        1948
2      128014
3       10286
4      172846
5        5228
6        5385
7        5931
8          11
9          36
10      90694
11       3219
12       1470
13         21
14        652
Name: count, dtype: int64


In [8]:
from src.models.isolation_forest import IsolationForestModel
from src.config.isolation_forest_config import IsolationForestConfig

config = IsolationForestConfig(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
)

model = IsolationForestModel(config)

print(model)

IsolationForestModel(n_estimators=200, contamination=0.05, is_fitted=False)


In [9]:


model.fit(X)

2026-07-28 15:06:48 | INFO     | src.models.isolation_forest | Training Isolation Forest...
2026-07-28 15:06:48 | INFO     | src.models.isolation_forest | Training samples: 2520798
2026-07-28 15:07:02 | INFO     | src.models.isolation_forest | Isolation Forest training completed.


IsolationForestModel(n_estimators=200, contamination=0.05, is_fitted=True)

In [10]:
predictions = model.predict(X)

print(predictions[:20])
print("Unique predictions:", set(predictions))

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Unique predictions: {np.int64(0), np.int64(1)}


In [11]:
scores = model.anomaly_score(X)

print("Min :", scores.min())
print("Max :", scores.max())
print("Mean:", scores.mean())

Min : 0.3199917115227662
Max : 0.7675161793144556
Mean: 0.38048771326207764


In [14]:
import pandas as pd

results = pd.DataFrame({
    "TrueLabel": y,
    "Prediction": predictions,
    "Score": scores
})

results.head()

,TrueLabel,Prediction,Score
0,0,0,0.381242
1,0,0,0.421825
4,0,0,0.384152
5,0,0,0.421407
8,0,0,0.345567


In [15]:
results.groupby("Prediction")["Score"].describe()

,count,mean,std,min,25%,50%,75%,max
Prediction,,,,,,,,
0,2394761.0,0.370103,0.054171,0.319992,0.330444,0.349820,0.392685,0.535368
1,126037.0,0.577808,0.039273,0.535369,0.545515,0.564669,0.602972,0.767516


In [16]:
results.groupby("TrueLabel")["Score"].mean().sort_values(ascending=False)

TrueLabel
8     0.702438
9     0.577121
5     0.517547
6     0.502845
4     0.482309
3     0.444668
2     0.422165
11    0.405712
7     0.373418
0     0.370241
12    0.366123
13    0.360268
14    0.359990
1     0.353360
10    0.342275
Name: Score, dtype: float64

In [17]:
import numpy as np

unique, counts = np.unique(predictions, return_counts=True)

print(dict(zip(unique, counts)))

{np.int64(0): np.int64(2394761), np.int64(1): np.int64(126037)}


In [19]:
from pathlib import Path

save_dir = Path("trained_models")
save_dir.mkdir(exist_ok=True)

model_path = save_dir / "isolation_forest.joblib"

model.save(model_path)

print(f"Model saved to: {model_path}")

2026-07-26 15:05:50 | INFO     | src.models.isolation_forest | Isolation Forest saved -> trained_models/isolation_forest.joblib


Model saved to: trained_models/isolation_forest.joblib


In [23]:
loaded_model = IsolationForestModel.load(model_path)

print(loaded_model)

2026-07-26 15:27:09 | INFO     | src.models.isolation_forest | Isolation Forest loaded <- trained_models/isolation_forest.joblib


IsolationForestModel(n_estimators=200, contamination=0.05, is_fitted=True)


In [24]:
X_test = X.head(1000)

In [25]:
pred_original = model.predict(X_test)
pred_loaded = loaded_model.predict(X_test)

import numpy as np

print(np.array_equal(pred_original, pred_loaded))

True


In [26]:
scores_original = model.anomaly_score(X_test)
scores_loaded = loaded_model.anomaly_score(X_test)

print(np.allclose(scores_original, scores_loaded))

True


In [27]:
print("Original Configuration")
print(model.config)

print()

print("Loaded Configuration")
print(loaded_model.config)

Original Configuration
IsolationForestConfig(n_estimators=200, contamination=0.05, max_samples='auto', max_features=1.0, bootstrap=False, random_state=42, n_jobs=-1, verbose=0)

Loaded Configuration
IsolationForestConfig(n_estimators=200, contamination=0.05, max_samples='auto', max_features=1.0, bootstrap=False, random_state=42, n_jobs=-1, verbose=0)


In [28]:
from pathlib import Path
from src.models.isolation_forest import IsolationForestModel

model_path = Path("trained_models/isolation_forest.joblib")

loaded_model = IsolationForestModel.load(model_path)

2026-07-26 15:27:13 | INFO     | src.models.isolation_forest | Isolation Forest loaded <- trained_models/isolation_forest.joblib


In [29]:
print(type(model.model))

AttributeError: 'IsolationForestModel' object has no attribute 'model'

In [30]:
pred_original = model.predict(X_test)
pred_loaded = loaded_model.predict(X_test)

print(np.array_equal(pred_original, pred_loaded))

True
